# FileID import for read/write

Start `examples/python/fileTransfer/testDevice.py` in another process, then run this notebook. It uploads a caller-local file or in-memory payload into the target device persistence manager and passes the returned `FileID` to channels that accept file values.

In [ ]:
from pathlib import Path

import stipy


config = stipy.Configuration({
    "Device Name": "FileID Import Notebook Client",
    "IP Address": "localhost",
    "Module": "0",
    "Target Server": "sr-magis/2/Frame2",
})

config.set("NetworkHub", "NameService", "192.168.88.252:2809")
config.set("omniORB", "traceLevel", "0")
config.set("omniORB", "scanGranularity", "1")
config.set("omniORB", "clientConnectTimeOutPeriod", "500")
config.set("omniORB", "clientCallTimeOutPeriod", "2000")

device_id = stipy.DeviceID("FileTransferDevice", "localhost", 0, "sr-magis/2/Frame2")
device = stipy.connect(device_id, config=config)
persistence = device.getPersistenceManager()
target_file_server = persistence.getFileServer()

assert persistence is not None
assert target_file_server is not None

device

Choose an existing file on the notebook machine. `Device.upload_file()` will create the source holder and file server, transfer the file into the target device, and return an `ImportedFile` context manager.

In [ ]:
source_path = Path("testDevice.py").resolve(strict=True)
source_size = source_path.stat().st_size

source_path, source_size

Keep the upload context open while its target-side `FileID` is in use. Exiting the context releases the imported registration and removes the temporary target file.

In [ ]:
with device.upload_file(source_path) as uploaded:
    imported_id = uploaded.fileID
    assert target_file_server.findFile(imported_id)
    assert target_file_server.getFileSize(imported_id) == source_size

    assert device.write(6, imported_id)
    assert device.read(20, imported_id) == source_size

assert uploaded.closed
assert not target_file_server.findFile(imported_id)

imported_id.filename, source_size

`Device.upload_data()` provides the same target-side `FileID` lifecycle for bytes-like data that is already in memory. This example requests virtual target storage, so neither the source nor the imported target payload needs a disk file.

In [ ]:
payload = bytearray(b"fileTransfer in-memory payload\n")
options = stipy.ImportFileOptions(storage=stipy.ImportStorage.Virtual)

with device.upload_data(memoryview(payload), "notebook-memory.txt", options=options) as uploaded:
    imported_id = uploaded.fileID
    assert target_file_server.findFile(imported_id)
    assert target_file_server.getFileSize(imported_id) == len(payload)

    assert device.write(6, imported_id)
    assert device.read(20, imported_id) == len(payload)

assert uploaded.closed
assert not target_file_server.findFile(imported_id)

imported_id.filename, len(payload)